<div style="display: flex; align-items: center;">
    <h1>Differentiable irrigation control with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

Differentiable control treats the crop model as the plant. A policy $\pi_\omega$ chooses an action $a_t$ each day; the process-based model $f_{\mathrm{PBM}}$ steps the crop and soil forward; a seasonal objective — here euro per hectare — is differentiated back through that trajectory into the policy weights $\omega$. Management is then an ordinary gradient step, on the same tape as parameter estimation or a hybrid module.

Irrigation is a natural first case. It is already part of the system dynamics of WOFOST, so an economic objective can send gradients through the water balance and into the rule that decided *when* to irrigate.

This notebook keeps water-limited WOFOST unchanged and injects a daily irrigation amount $a_t$ (mm) into the freely draining water balance. Four controllers are compared on the same euro-per-hectare reward:

- **rainfed**: no irrigation;
- **potential production**: soil moisture held at field capacity, no irrigation bill;
- **AIMCRA**: the Spanish grower water-balance / NAP rule, not trained;
- a **random-init MLP** trained with a weekly decision grid, then deployed every day.

$$
s_{t+1} = f_{\mathrm{PBM}}\bigl(s_t, x_t, a_t, \theta\bigr), \qquad a_t = \pi_\omega(s_t, x_t).
$$

Here $s_t$ is the crop and soil state, $x_t$ is the weather, $\theta$ are the usual WOFOST parameters, and $\omega$ are the MLP weights. The reward is beet revenue minus irrigation cost. Controllers are fit on the 2010 YAML weather year at the Spanish test site (39.30°N, 3.43°W) with the YAML soil; a later section freezes those weights and evaluates them on NASA POWER 2009 and 2011 at the same coordinates.


## 1. Why differentiable control is useful

This control perspective has three practical advantages.

1. **Transparency.** The consequences of an irrigation decision can be traced through soil moisture, transpiration reduction (`RFTRA`), assimilation, and storage-organ growth. That is a more mechanistic account than optimizing a policy against an opaque reward.

2. **Data efficiency.** Gradient-based updates provide a directional signal for changing the policy. The optimizer does not have to discover useful irrigations by randomly exploring state–action trajectories.

3. **Coupling to physiology.** The policy remains inside the water balance, so it is automatically constrained by infiltration capacity, percolation, crop presence, and the rest of the WOFOST dynamics.

There are also real difficulties, which this notebook will run into in miniature. Early-season irrigations affect yield months later, so the gradient signal can be weak. Spanish sugar-beet irrigation is *event-based* — AIMCRA's in-season gifts are about 40 mm on a loam — whereas a naive differentiable controller emits a continuous daily trickle. A straight-through estimator makes the forward pass look like a farmer event while Adam still receives a gradient.

Daily decisions from a random MLP are too sparse for that tape: the €25 event fee plus a hard $p>0.5$ gate walks the policy to rainfed before it ever finds AIMCRA's calendar. The workable inductive bias here is **time**, not an agronomic prior. Train the same MLP on a weekly grid, keep the best hard $R$, then **deploy those weights every day**. Reinforcement learning on the same daily problem is left for later.


## 2. Software requirements

Install the latest `diffwofost` if needed. The notebook also uses `pandas` and `matplotlib`.
Crop, soil and 2010 weather come from PCSE's water-limited sugar-beet YAML
(`test_waterlimitedproduction_wofost72_03.yaml`: 39.30°N, 3.43°W, 629 m).
Held-out years use NASA POWER at the same site.


In [ ]:
%%capture
# install required packages when needed
!pip install -q diffwofost matplotlib pandas


In [1]:
%matplotlib inline

from pathlib import Path
import copy
import datetime as dt
import urllib.request
import warnings

import matplotlib
matplotlib.style.use("ggplot")
import matplotlib.pyplot as plt
import pandas as pd
import torch
from pcse.input import NASAPowerWeatherDataProvider
from pcse.exceptions import PCSEError

from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalanceFD, WaterbalancePP

try:
    from diffwofost.physical_models.test import get_test_data, prepare_engine_input
except ModuleNotFoundError:
    # PyPI 0.5.0 still exports these from utils; they moved to test.py after that release.
    from diffwofost.physical_models.utils import get_test_data, prepare_engine_input

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

print(f"torch version: {torch.__version__}")
print(f"device: {ComputeConfig.get_device()}")
print(f"dtype: {ComputeConfig.get_dtype()}")


torch version: 2.11.0+cu130
device: cpu
dtype: torch.float64


## 3. A water-limited sugar-beet season

We use PCSE's water-limited WOFOST 7.2 regression case `test_waterlimitedproduction_wofost72_03.yaml`: sugar beet sown on 27 March 2010 at **39.30°N, 3.43°W**, elevation 629 m (Castilla-La Mancha), with the YAML's own soil (`SMW`/`SMFCF`/`SM0` = 0.152 / 0.318 / 0.416). That is the original test case. The freely draining water balance (`WaterbalanceFD`) has no capillary rise from groundwater.

The AIMCRA 40 mm gift is the *event size* prior. For this province and sowing date, a mean-year calendar already has about 15 events and ~540 mm net.

Two reference runs bound the problem:

- **Rainfed water-limited production**: no irrigation.
- **Potential production**: the same crop with soil moisture held at field capacity. This is the physiological yield ceiling if water were never limiting, and it is *not* an irrigated operating plan (the irrigation bill is zero by construction).

WOFOST reports `TWSO` as kg/ha of *dry* storage-organ biomass. Fresh beet yield is obtained with a dry-matter fraction of 0.23:

$$
Y_{\mathrm{fresh}} = \frac{\mathrm{TWSO}}{1000 \times 0.23}\quad[\mathrm{t/ha}].
$$


In [ ]:
def download_if_needed(path, url):
    path = Path(path)
    if path.exists():
        print(f"Using local file: {path}")
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path.name}")
    return path


filename = "test_waterlimitedproduction_wofost72_03.yaml"
test_data_path = download_if_needed(
    filename,
    "https://raw.githubusercontent.com/ajwdewit/pcse/refs/heads/master/"
    f"tests/test_data/{filename}",
)
test_data = get_test_data(test_data_path)

crop_model_params = [
    "SPAN", "TDWI", "TBASE", "PERDL", "RGRLAI", "KDIFTB", "SLATB",
    "TSUMEM", "TBASEM", "TEFFMX", "TSUM1", "TSUM2", "DLO", "DLC", "DVSI", "DVSEND", "DTSMTB",
    "AMAXTB", "EFFTB", "TMPFTB", "TMNFTB",
    "Q10", "RMR", "RML", "RMS", "RMO", "RFSETB",
    "CFET", "DEPNR", "IAIRDU", "IOX", "CRAIRC", "SM0", "SMW", "SMFCF", "WAV",
    "RDI", "RRI", "RDMCR", "RDMSOL", "RDRRTB",
    "RDRSTB", "SSATB", "SPA",
    "FRTB", "FLTB", "FSTB", "FOTB",
    "CVL", "CVO", "CVR", "CVS",
    "SOPE", "KSUB", "SMLIM",
]
provider, weather, yaml_agro, _ = prepare_engine_input(test_data, crop_model_params)
agromanagement = yaml_agro

TRAIN_YEAR = 2010
TEST_YEARS = (2009, 2011)

campaign_start = next(iter(agromanagement[0].keys()))
site = weather(campaign_start)
SITE_LAT, SITE_LON, SITE_ELEV = float(site.LAT), float(site.LON), float(site.ELEV)


def agro_for_year(agromanagement, year):
    shifted = []
    for campaign in copy.deepcopy(agromanagement):
        new_campaign = {}
        for start, spec in campaign.items():
            calendar = spec["CropCalendar"]
            for key in ("crop_start_date", "crop_end_date"):
                calendar[key] = calendar[key].replace(year=year)
            new_campaign[start.replace(year=year)] = spec
        shifted.append(new_campaign)
    return shifted


class NASAPowerWindow(NASAPowerWeatherDataProvider):
    """NASA POWER with a finite past window.

    PCSE's default query uses ``user=anonymous`` and ``end=today``, which the
    POWER API currently rejects with HTTP 422.
    """

    def __init__(self, latitude, longitude, start, end, **kwargs):
        self._window_start = start
        self._window_end = end
        super().__init__(latitude, longitude, **kwargs)

    def _query_NASAPower_server(self, latitude, longitude):
        import requests

        server = "https://power.larc.nasa.gov/api/temporal/daily/point"
        payload = {
            "request": "execute",
            "parameters": ",".join(self.power_variables),
            "latitude": latitude,
            "longitude": longitude,
            "start": self._window_start.strftime("%Y%m%d"),
            "end": self._window_end.strftime("%Y%m%d"),
            "community": "AG",
            "format": "JSON",
        }
        req = requests.get(server, params=payload)
        if req.status_code != self.HTTP_OK:
            raise PCSEError(
                f"Failed retrieving POWER data, server returned HTTP {req.status_code} "
                f"on {req.url}"
            )
        return req.json()


print(f"crop: {test_data['ModelParameters'].get('CRPNAM', 'unknown')}")
print("soil: YAML (original test file)")
print(
    f"weather: YAML {TRAIN_YEAR}  {SITE_LAT:.5f}°N, {abs(SITE_LON):.5f}°W  "
    f"elev {SITE_ELEV:.0f} m  (Castilla-La Mancha)"
)
print(f"campaign year: {TRAIN_YEAR} (train); NASA POWER held-out years {TEST_YEARS}")
print(f"WAV (initial available water): {float(provider['WAV']):.1f} cm")
print(
    "soil moisture bounds SMW / SMFCF / SM0: "
    f"{float(provider['SMW']):.3f} / {float(provider['SMFCF']):.3f} / {float(provider['SM0']):.3f}"
)


In [ ]:
wlp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalanceFD,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "TRA", "RFTRA", "TWSO", "TWST", "RD"],
)

pp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "RFTRA", "TWSO"],
)


def scalarize(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu())
    return value


def results_to_frame(results, irrigation=None):
    rows = []
    for row in results:
        rows.append({key: (value if key == "day" else scalarize(value)) for key, value in row.items()})
    frame = pd.DataFrame(rows)
    frame["day"] = pd.to_datetime(frame["day"])
    frame = frame.set_index("day")
    if irrigation is not None:
        irr = pd.Series(
            {day: 10.0 * scalarize(amount) for day, amount in irrigation},
            name="irrigation",
        )
        irr.index = pd.to_datetime(irr.index)
        frame = frame.join(irr, how="left")
        frame["irrigation"] = frame["irrigation"].fillna(0.0)
    return frame


def summarize(name, frame, totirr=0.0):
    emerged = frame[frame["DVS"] > 0]
    n_irr = int((frame.get("irrigation", pd.Series(0, index=frame.index)) > 15.0).sum())
    print(
        f"{name:22s}  TWSO={frame['TWSO'].iloc[-1]:8.1f} kg/ha  "
        f"TOTIRR={totirr:5.1f} cm  min RFTRA={emerged['RFTRA'].min():.2f}  "
        f"events>15 mm={n_irr:3d}"
    )


rainfed_engine = Engine(config=wlp_config)
rainfed_engine.setup(provider, weather, agromanagement)
rainfed_engine.run_till_terminate()
rainfed_df = results_to_frame(rainfed_engine.get_output())

pp_engine = Engine(config=pp_config)
pp_engine.setup(provider, weather, agromanagement)
pp_engine.run_till_terminate()
pp_df = results_to_frame(pp_engine.get_output())

summarize("rainfed", rainfed_df, totirr=0.0)
summarize("potential production", pp_df, totirr=float("nan"))
print(
    "yield gap due to water: "
    f"{pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]:.0f} kg/ha dry  "
    f"({(pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]) / 1000 / 0.23:.1f} t/ha fresh at 23% DM)"
)


The rainfed crop loses a large fraction of potential storage-organ biomass. Soil moisture falls well below field capacity during canopy expansion, `RFTRA` drops, and growth of the beet (`TWSO`) stalls. Closing that gap is valuable only when the extra beet revenue exceeds the cost of the irrigation events that produced it. A fixed calendar such as "irrigate every 7 days" is a poor abstraction here: rainfall, soil water-holding capacity and crop demand jointly determine whether the next event is needed at all.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)

axes[0].plot(rainfed_df.index, rainfed_df["SM"], label="rainfed")
axes[0].plot(pp_df.index, pp_df["SM"], label="potential", linestyle="--")
axes[0].axhline(float(provider["SMFCF"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].axhline(float(provider["SMW"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("SM $(-)$")
axes[0].legend()

axes[1].plot(rainfed_df.index, rainfed_df["RFTRA"], label="rainfed")
axes[1].plot(pp_df.index, pp_df["RFTRA"], label="potential", linestyle="--")
axes[1].set_ylabel("RFTRA $(-)$")

axes[2].plot(rainfed_df.index, rainfed_df["TWSO"], label="rainfed")
axes[2].plot(pp_df.index, pp_df["TWSO"], label="potential", linestyle="--")
axes[2].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[2].set_xlabel("day")

fig.suptitle("Water-limited rainfed crop versus potential production", y=0.99)
fig.tight_layout()
plt.show()


## 4. Irrigation inside the water balance

In `WaterbalanceFD`, an irrigation event sets the effective irrigation rate `_RIRR` (applied amount times application efficiency). That rate is added to infiltrating water, updates root-zone moisture `SM`, and therefore changes the transpiration reduction factor `RFTRA` used by assimilation.

PCSE normally fires irrigation through an agromanagement signal. Here we do the same thing from Python: after the daily weather and management step, and before `calc_rates`, the policy writes a tensor into `engine.soil._RIRR`. Because that tensor is produced by $\pi_\omega$, backpropagation can flow from final `TWSO` through the water balance and into the policy weights.

Application efficiency is set to 0.8, AIMCRA's usual figure for sprinkler *cobertura total* (the handbook range is 0.75–0.90; drip is 0.90–0.95 and is not the baseline). The policy and the cost model are expressed in **applied millimetres** (what the farmer decides). `_RIRR` and `TOTIRR` use WOFOST's centimetre units; `TOTIRR` is the *effective* amount that entered the soil.


## 5. Event-based irrigation as an action

Realistic sugar-beet irrigation is not a continuous water supply. This YAML is a Spanish spring crop at **39.30°N, 3.43°W** (Ciudad Real / Castilla-La Mancha). The grower handbook is AIMCRA's [*Técnicas de riego en la remolacha azucarera*](https://www.aimcra.es/publicaciones/documentos/otras/tecnicas_de_riego.pdf) (2001): water-balance scheduling.

AIMCRA treats the soil as a store. Only a fraction of plant-available water is allowed to go (NAP, *nivel de agotamiento permitido*: about 30% on clay, 70% on sand). UPM work cited there puts the yield optimum at **45 cb at 30 cm**, which is about **35 mm** of depletion. Recommended in-season gifts on the northern/central loams are **38–44 mm** (40 mm on a clay loam). Peak-month CROPWAT calendars use a **40 mm net** pulse. Sprinkler efficiency for a well-run solid set is **0.8**.

Sowing on 27 March is AIMCRA's *siembra tardía* (after 15 March). The mean-year net calendar for **Ciudad Real** on a soil with 140–180 mm m$^{-1}$ of available water — the YAML loam is 166 mm m$^{-1}$ — is **14–15 events and ~535–540 mm net** (May–September). Crop ET for the same late sowing is 634 mm (May–September). Drip is not the Spanish beet baseline.

```
soil water
100% |───────╮            ╭────────╮
             │            │        │
             ╰────────────╯        ╰──────
             ↑            ↑
          40 mm        40 mm
        AIMCRA gift   next pulse
```

Irrigation scientists call the soil-moisture version **management allowed depletion** (MAD); AIMCRA's name is the water-balance / NAP rule. `StandardPracticePolicy` is that handbook **frozen**: irrigate when about 35 mm is gone from the current root zone, never more than 65% of plant-available water, apply 50 mm (40 mm net at 80% efficiency) capped by the root-zone remainder, skip rain ≥ 10 mm, after emergence, last irrigation 15 September. It is not trained. Emergence irrigations (spring: 20–25 mm, then 5–10 mm) are a separate seedbed protocol and are omitted.

The learned controller is a small MLP on the **same information class** as AIMCRA: refill-to-field-capacity remainder, today's rain, DVS, and day of year. It does not see `RFTRA`. The gift is the same 50 mm applied pulse, clipped to the remainder; the event is skipped only if that remainder is below 1 mm. A weekly mask `(DOY − 1) mod 7 = 0` is used **during training**. At deployment the mask is removed and the same weights decide every day.


In [ ]:
IRRIGATION_EFFICIENCY = 0.8  # AIMCRA aspersión, cobertura total
DM_FRACTION = 0.23           # fresh beet (t/ha) = TWSO (kg/ha) / 1000 / DM
BEET_PRICE = 40.0            # € / t fresh; typical NW-EU contract (2024 ~€38–47)
C_MM = 1.0                   # € / mm / ha for water + pumping energy
C_EVENT = 25.0               # € / ha labour/tractor to apply one hose-reel set
EVENT_MM_MIN = 1.0           # mobilisation billed for any real application
EVENT_TEMP_MM = 0.1          # 0 mm → ~0 events; 1 mm → half; ≥2 mm → a full fee
MLP_DOSE_MM = 50.0           # AIMCRA 40 mm net / 50 mm applied
MLP_HIDDEN = 64
MLP_N_OBS = 4                # cap, rain, DVS, DOY — no RFTRA


def kiosk_get(engine, name, default):
    if name in engine.kiosk:
        value = engine.kiosk[name]
        if value is not None:
            return value
    return default


def as_scalar(value, like):
    tensor = torch.as_tensor(value, dtype=like.dtype, device=like.device)
    return tensor.reshape(()) if tensor.numel() == 1 else tensor


def stacked_amounts(irrigation):
    return torch.stack([amount for _, amount in irrigation])


def fresh_yield_t_ha(twso):
    return twso / 1000.0 / DM_FRACTION


def irrigation_economics(twso, amounts_cm, n_events=None, c_event=None, c_mm=None):
    """R = P Y - C_event N - c_mm I. N is any day with applied water."""
    y_fresh = fresh_yield_t_ha(twso)
    revenue = BEET_PRICE * y_fresh
    applied_mm = amounts_cm * 10.0
    if n_events is None:
        n_events = torch.sigmoid((applied_mm - EVENT_MM_MIN) / EVENT_TEMP_MM).sum()
    c_event = C_EVENT if c_event is None else c_event
    c_mm = C_MM if c_mm is None else c_mm
    cost = c_event * n_events + c_mm * applied_mm.sum()
    reward = revenue - cost
    n_hard = int((applied_mm.detach() > EVENT_MM_MIN).sum())
    return {
        "y_fresh": y_fresh,
        "revenue": revenue,
        "cost": cost,
        "reward": reward,
        "applied_mm": applied_mm.sum(),
        "n_events": n_events,
        "n_events_hard": n_hard,
    }


def is_decision_day(doy, period):
    if period <= 1:
        return True
    return ((int(doy) - 1) % int(period)) == 0


def twohead_inputs(engine):
    """Closed-loop features in AIMCRA's information class: cap, rain, DVS, DOY.

    `applied_cap_mm` is the applied millimetres that would refill the current
    root zone to field capacity at 80% efficiency. The gate may fire a 50 mm
    AIMCRA gift, but applied water is always min(50 mm, that remainder).
    """
    sm = engine.soil.states.SM
    smfc = engine.soil.params.SMFCF
    rain = as_scalar(engine.drv.RAIN, sm)
    dvs = as_scalar(kiosk_get(engine, "DVS", torch.zeros_like(sm)), sm)
    rd = as_scalar(
        kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
        sm,
    ).clamp_min(1.0)
    doy_i = engine.day.timetuple().tm_yday
    doy = torch.as_tensor(float(doy_i), dtype=sm.dtype, device=sm.device)
    deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
    applied_cap_mm = deficit_cm / IRRIGATION_EFFICIENCY * 10.0
    x = torch.stack(
        [
            applied_cap_mm / 50.0,
            rain,
            dvs / 2.0,
            doy / 365.0,
        ]
    )
    return x, applied_cap_mm, dvs


class StandardPracticePolicy(torch.nn.Module):
    """AIMCRA *Técnicas de riego* (2001). Not trained.

    Single-layer water balance: irrigate when about 35 mm of readily
    available water has gone from the current root zone (UPM 45 cb at
    30 cm), but never more than 65% of plant-available water in that
    depth. Apply a 40 mm *net* gift (50 mm applied at 80% efficiency),
    capped by the root-zone deficit. Skip a useful rain day. After
    emergence; last irrigation 15 September (AIMCRA cut-off for
    October / November lifting). Emergence irrigations are omitted.
    """

    def __init__(
        self,
        net_dose_mm=40.0,
        nap_mm=35.0,
        nap_frac=0.65,
        useful_rain_mm=10.0,
        last_month=9,
        last_day=15,
        efficiency=None,
    ):
        super().__init__()
        self.net_dose_mm = net_dose_mm
        self.nap_mm = nap_mm
        self.nap_frac = nap_frac
        self.useful_rain_mm = useful_rain_mm
        self.last_month = last_month
        self.last_day = last_day
        self.efficiency = IRRIGATION_EFFICIENCY if efficiency is None else efficiency

    def reset(self):
        return None

    def forward(self, engine):
        sm = engine.soil.states.SM
        zero = torch.zeros((), dtype=sm.dtype, device=sm.device)
        day = engine.day
        if (day.month, day.day) > (self.last_month, self.last_day):
            return zero

        rain_mm = float(as_scalar(engine.drv.RAIN, sm).detach()) * 10.0
        dvs = float(as_scalar(kiosk_get(engine, "DVS", zero), sm).detach())
        if dvs <= 0.0 or rain_mm >= self.useful_rain_mm:
            return zero

        smw = engine.soil.params.SMW
        smfc = engine.soil.params.SMFCF
        rd = as_scalar(
            kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
            sm,
        ).clamp_min(1.0)
        rd_cm = float(rd.detach())
        available_mm = max(float((smfc - smw).detach()) * rd_cm * 10.0, 1.0)
        nap_mm = min(self.nap_mm, self.nap_frac * available_mm)
        depletion_mm = float((smfc - sm).clamp(min=0.0).detach()) * rd_cm * 10.0
        if depletion_mm < nap_mm:
            return zero

        deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
        applied_cap = deficit_cm / self.efficiency
        target_cm = self.net_dose_mm / 10.0 / self.efficiency
        applied_cm = torch.minimum(
            applied_cap, torch.as_tensor(target_cm, dtype=sm.dtype, device=sm.device)
        )
        if float(applied_cm.detach()) * 10.0 < EVENT_MM_MIN:
            return zero
        return applied_cm


class WeeklyAIMCRA(torch.nn.Module):
    """Same AIMCRA threshold, allowed only on a regular calendar grid."""

    def __init__(self, period=7):
        super().__init__()
        self.inner = StandardPracticePolicy()
        self.period = int(period)

    def reset(self):
        return self.inner.reset()

    def forward(self, engine):
        sm = engine.soil.states.SM
        zero = torch.zeros((), dtype=sm.dtype, device=sm.device)
        if not is_decision_day(engine.day.timetuple().tm_yday, self.period):
            return zero
        return self.inner(engine)


class IrrigationMLP(torch.nn.Module):
    """Random-init MLP gate on AIMCRA's features.

    Train with ``period=7`` (one decision per week). Deploy with ``period=1``
    (the same weights, every day). Forward pass is a hard event when
    ``p > 0.5``; backward pass uses a straight-through sigmoid. Applied
    water is ``min(50 mm, refill-to-FC remainder)``; skip only if that
    remainder is below 1 mm.
    """

    def __init__(self, hidden=MLP_HIDDEN, dose_mm=MLP_DOSE_MM, period=7):
        super().__init__()
        dtype = ComputeConfig.get_dtype()
        device = ComputeConfig.get_device()
        self.dose_cm = dose_mm / 10.0
        self.period = int(period)
        self.hard_decisions = False
        self.body = torch.nn.Sequential(
            torch.nn.Linear(MLP_N_OBS, hidden, dtype=dtype, device=device),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden, dtype=dtype, device=device),
            torch.nn.Tanh(),
        )
        self.actor = torch.nn.Linear(hidden, 1, dtype=dtype, device=device)
        self.event_terms = []
        self.p_terms = []

    def reset(self):
        self.event_terms = []
        self.p_terms = []

    def forward(self, engine):
        sm = engine.soil.states.SM
        zero = torch.zeros((), dtype=sm.dtype, device=sm.device)
        if not is_decision_day(engine.day.timetuple().tm_yday, self.period):
            return zero
        x, cap_mm, dvs = twohead_inputs(engine)
        logit = self.actor(self.body(x)).reshape(())
        p = torch.sigmoid(logit)
        crop = (dvs > 0).to(dtype=sm.dtype)
        if self.hard_decisions:
            hard = (p > 0.5).to(dtype=p.dtype)
            gate = crop * (hard.detach() + p - p.detach())
        else:
            gate = crop * p
        dose = torch.as_tensor(self.dose_cm, dtype=sm.dtype, device=sm.device)
        applied = torch.minimum(dose, cap_mm / 10.0)
        gate = gate * (1.0 - (applied * 10.0 < EVENT_MM_MIN).to(dtype=sm.dtype))
        self.event_terms.append(gate)
        self.p_terms.append(crop * p)
        return gate * applied


def run_with_policy(engine, policy=None, efficiency=IRRIGATION_EFFICIENCY):
    """Advance a prepared engine, injecting irrigation (cm) before each rate calculation."""
    zero = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    irrigation = []
    while engine.flag_terminate is False:
        engine.day, delt = engine.timer()
        engine.integrate(engine.day, delt)
        engine.drv = engine._get_driving_variables(engine.day)
        engine.agromanager(engine.day, engine.drv)
        amount_cm = zero if policy is None else policy(engine)
        engine.soil._RIRR = amount_cm * efficiency
        irrigation.append((engine.day, amount_cm))
        engine.calc_rates(engine.day, engine.drv)
        if engine.flag_terminate is True:
            engine._terminate_simulation(engine.day)
    return irrigation


def evaluate_policy(policy, provider, weather, agromanagement, config, c_event=None):
    if policy is not None and hasattr(policy, "reset"):
        policy.reset()
    engine = Engine(config=config)
    engine.setup(provider, weather, agromanagement)
    irrigation = run_with_policy(engine, policy=policy)
    results = engine.get_output()
    amounts_cm = stacked_amounts(irrigation)
    twso = results[-1]["TWSO"]
    totirr = engine.soil.states.TOTIRR
    n_events = None
    if policy is not None and getattr(policy, "event_terms", None):
        n_events = torch.stack(policy.event_terms).sum()
    eco = irrigation_economics(twso, amounts_cm, n_events=n_events, c_event=c_event)
    return {
        "engine": engine,
        "results": results,
        "frame": results_to_frame(results, irrigation),
        "irrigation": irrigation,
        "amounts_cm": amounts_cm,
        "twso": twso,
        "totirr": totirr,
        **eco,
    }


The daily loop is the ordinary PCSE step, with one extra line that writes $a_t = \pi_\omega(s_t, x_t)$ into the water balance. That is the implementation of the decision-making box: the policy is inside $f_{\mathrm{PBM}}$, not wrapped around it as an external simulator. `_RIRR` is in cm; plots and the cost model convert to mm (1 mm = 10 m³/ha).


In [ ]:
def summarize_economics(name, outcome=None, frame=None, amounts_cm=None, potential=False):
    if frame is None:
        frame = outcome["frame"]
    twso = frame["TWSO"].iloc[-1] if outcome is None else outcome["twso"]
    if potential:
        y = scalarize(fresh_yield_t_ha(twso))
        print(
            f"{name:28s}  fresh={y:5.1f} t/ha  applied=   0 mm  events=  0  "
            f"revenue={BEET_PRICE * y:7.0f} €/ha  cost=    0  reward={BEET_PRICE * y:7.0f} €/ha"
        )
        return
    if outcome is not None:
        eco = outcome
        amounts = outcome["amounts_cm"]
    else:
        amounts = amounts_cm
        eco = irrigation_economics(torch.as_tensor(twso, dtype=ComputeConfig.get_dtype()), amounts)
    y = scalarize(eco["y_fresh"])
    mm = scalarize(eco["applied_mm"])
    n_hard = eco["n_events_hard"] if outcome is not None else int((amounts.detach() * 10 > EVENT_MM_MIN).sum())
    print(
        f"{name:28s}  fresh={y:5.1f} t/ha  applied={mm:5.0f} mm  events={n_hard:3d}  "
        f"revenue={scalarize(eco['revenue']):7.0f} €/ha  "
        f"cost={scalarize(eco['cost']):5.0f}  reward={scalarize(eco['reward']):7.0f} €/ha"
    )


zero_amounts = torch.zeros(len(rainfed_df), dtype=ComputeConfig.get_dtype())
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("potential production", frame=pp_df, potential=True)

standard_practice = evaluate_policy(
    StandardPracticePolicy(), provider, weather, agromanagement, wlp_config
)
print(
    "standard practice (AIMCRA): 40 mm net when ~35 mm of root-zone water "
    "is gone, after emergence, last irrigation 15 September"
)
summarize_economics("standard practice (AIMCRA)", standard_practice)


## 6. An economic reward, not a water penalty

Irrigation is not inherently bad. AIMCRA lists several programming objectives; one of them is to **maximise farm profit**. The agent should irrigate when the extra beet value is expected to exceed the cost of the event.

Splitting the bill into a mobilisation fee and a cheaper volume term is closer to how the cost actually arises:

$$
C(I, N) = C_{\mathrm{event}}\,N + c_{\mathrm{mm}}\,I,
\qquad
R = P\,Y_{\mathrm{fresh}} - C(I, N).
$$

| assumption | value | reason |
|---|---|---|
| beet price `P` | €40/t | typical NW-EU contract; 2024 was about €38–47, high years ~€50 |
| volume cost `c_mm` | €1.0 per mm per ha | water + pumping (1 mm = 10 m³/ha) |
| event cost `C_event` | €25/ha | labour/tractor to apply one set |
| dry-matter fraction | 0.23 | WOFOST `TWSO` is dry biomass |

A 25 mm application then costs €25 + €25 = €50/ha. An AIMCRA-sized 50 mm applied gift (40 mm net) costs €25 + €50 = €75/ha. The €25 is billed on **any day that water is applied** — going out with the reel — not only after 5 or 15 mm. A 1 mm numerical floor keeps a true zero day at $N=0$ on the sigmoid. Capital (borehole, pump, pipes, reel) is excluded: it is a sunk cost if the kit is already there.

At 80% efficiency, 50 mm applied is 40 mm effective. Extra beet only has to cover €1.25 of volume cost per effective millimetre, plus the event fee, *when there is a real deficit*.

Adam minimizes $J = -R$. Comparison tables report $R$ in €/ha. Potential production is listed as beet *revenue* with a zero irrigation bill.

Because $Y$, $I$ and $N$ are all produced on the same gradient tape as the water balance, a standard Adam step on $\omega$ is enough — provided the decision grid is coarse enough for the straight-through gate to find events.


## 7. Weekly training, daily deployment

A random MLP on **daily** decisions does not discover AIMCRA. Default Linear init has $\bar p \approx 0.44$; the hard gate (`p > 0.5`) never turns on, and STE plus a €25 jump walks the few ON days to rainfed. That is a property of the daily combinatorial problem, not of differentiability.

The same net on a **weekly** grid has ~40 candidate days instead of ~280. Soft warmup (40 Adam steps) raises $p$ enough that STE can fire events; further STE then matches or slightly beats weekly AIMCRA on $R$. Features stay in AIMCRA's class: remainder cap, rain, DVS, day of year. No `RFTRA`.

Daily **deployment** is not a second training problem. Set `period=1` on the weekly peak weights. After a 50 mm gift the next-day refill cap is small, so neighbouring days stay OFF: the weekly calendar does not flood when read every day. Daily STE finetuning of those weights can improve $R$ for a step or two and then collapses toward rainfed; this notebook keeps the weekly checkpoint and does not finetune on the daily tape. Reinforcement learning on daily actions is left for later.

Training: 40 soft weekly steps, 80 STE steps, 120 lower-lr STE steps. The best *hard* $R$ is kept (the surrogate tape can walk off a good discrete schedule). If `data_temp/mlp_weekly_best.pt` already exists, that cell loads it instead of retraining.


In [ ]:
ckpt_dir = Path("data_temp")
ckpt_dir.mkdir(exist_ok=True)
CKPT_WEEKLY = ckpt_dir / "mlp_weekly_best.pt"
CKPT_LEGACY = ckpt_dir / "mlp_aimcra_norftra_best.pt"


def clone_state(net):
    return {key: value.detach().clone() for key, value in net.state_dict().items()}


def hard_eval(net, weather_provider=weather, agro=agromanagement):
    was = net.hard_decisions
    net.hard_decisions = True
    with torch.no_grad():
        out = evaluate_policy(net, provider, weather_provider, agro, wlp_config)
    net.hard_decisions = was
    return out


def load_checkpoint(path):
    try:
        blob = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        blob = torch.load(path, map_location="cpu")
    if isinstance(blob, dict) and blob.get("best_weekly") is not None:
        state, hist, saved_r = blob["best_weekly"], blob.get("history", []), blob.get("best_w_r")
    elif isinstance(blob, dict) and "state_dict" in blob:
        state, hist, saved_r = blob["state_dict"], blob.get("history", []), blob.get("best_r")
    else:
        state, hist, saved_r = blob, [], None
    weight = state.get("body.0.weight") if isinstance(state, dict) else None
    in_dim = int(weight.shape[1]) if weight is not None else None
    if in_dim != MLP_N_OBS:
        print(f"skip {path.name}: first layer is {in_dim} features, need {MLP_N_OBS}")
        return None
    hist = [row for row in (hist or []) if str(row.get("phase", "w")).startswith("w")]
    return state, hist, saved_r


weekly_aimcra = evaluate_policy(WeeklyAIMCRA(period=7), provider, weather, agromanagement, wlp_config)
print("Weekly AIMCRA is the same handbook rule, restricted to (DOY-1) mod 7 == 0.")
summarize_economics("AIMCRA weekly (diagnostic)", weekly_aimcra)

mlp = IrrigationMLP(period=7)
history = []
best_weekly = None
best_weekly_r = float("-inf")
loaded = False

for path in (CKPT_WEEKLY, CKPT_LEGACY):
    if not path.exists():
        continue
    loaded_blob = load_checkpoint(path)
    if loaded_blob is None:
        continue
    state, history, saved_r = loaded_blob
    mlp.load_state_dict(state)
    best_weekly = clone_state(mlp)
    best_weekly_r = float("-inf") if saved_r is None else float(saved_r)
    loaded = True
    print(f"loaded weekly MLP from {path}")
    torch.save(
        {"state_dict": clone_state(mlp), "history": history, "best_r": best_weekly_r},
        CKPT_WEEKLY,
    )
    break

if not loaded:
    torch.manual_seed(11)
    mlp = IrrigationMLP(period=7)
    with torch.no_grad():
        soft0 = evaluate_policy(mlp, provider, weather, agromanagement, wlp_config)
        p0 = torch.stack(mlp.p_terms)
        hard0 = hard_eval(mlp)
    print(
        f"random init  weekly  p̅={float(p0.mean()):.2f}  p>0.5={float((p0 > 0.5).float().mean()):.2f}\n"
        f"  soft R={scalarize(soft0['reward']):.0f}  hard R={scalarize(hard0['reward']):.0f}  "
        f"hard events={hard0['n_events_hard']}"
    )

    def run_weekly_phase(n_steps, lr, use_hard, tag, every=5):
        global best_weekly, best_weekly_r
        mlp.period = 7
        opt = torch.optim.Adam(mlp.parameters(), lr=lr)
        print(f"\n=== {tag}  period=7d  hard={use_hard}  {n_steps} steps  lr={lr} ===")
        for step in range(n_steps):
            opt.zero_grad()
            mlp.hard_decisions = use_hard
            train_out = evaluate_policy(mlp, provider, weather, agromanagement, wlp_config)
            pre = clone_state(mlp)
            (-train_out["reward"]).backward()
            torch.nn.utils.clip_grad_norm_(mlp.parameters(), 0.5)
            opt.step()
            if use_hard:
                hard = train_out
                saved = pre
            else:
                hard = hard_eval(mlp)
                saved = clone_state(mlp)
            rec_r = scalarize(hard["reward"])
            mean_p = float(torch.stack(mlp.p_terms).mean().detach()) if mlp.p_terms else 0.0
            tag_best = ""
            if rec_r > best_weekly_r:
                best_weekly_r = rec_r
                best_weekly = saved
                tag_best = "  BEST"
            rec = {
                "step": len(history),
                "phase": tag,
                "reward_hard": rec_r,
                "reward_train": scalarize(train_out["reward"]),
                "y_fresh": scalarize(hard["y_fresh"]),
                "applied_mm": scalarize(hard["applied_mm"]),
                "n_events": hard["n_events_hard"],
                "mean_p": mean_p,
            }
            history.append(rec)
            if step % every == 0 or tag_best or step < 5:
                print(
                    f"{tag} {rec['step']:03d}  p̅={mean_p:.2f}  "
                    f"fresh={rec['y_fresh']:5.1f} t/ha  applied={rec['applied_mm']:5.0f} mm  "
                    f"events={rec['n_events']:3d}  R_hard={rec_r:7.0f}{tag_best}",
                    flush=True,
                )

    run_weekly_phase(40, 1e-3, use_hard=False, tag="w-soft", every=5)
    run_weekly_phase(80, 1e-3, use_hard=True, tag="w-ste", every=5)
    if best_weekly is not None:
        mlp.load_state_dict(best_weekly)
        print(f"\nreload weekly peak R={best_weekly_r:.0f} €/ha")
    run_weekly_phase(120, 1e-4, use_hard=True, tag="w-fine", every=10)
    if best_weekly is not None:
        mlp.load_state_dict(best_weekly)
    torch.save({"state_dict": clone_state(mlp), "history": history, "best_r": best_weekly_r}, CKPT_WEEKLY)
    print(f"saved {CKPT_WEEKLY}")

mlp.period = 7
mlp.hard_decisions = True
weekly_mlp = hard_eval(mlp)

mlp.period = 1
daily_mlp = hard_eval(mlp)
print(
    "\nDaily deployment uses the weekly weights with period=1. "
    "There is no extra Adam on daily decisions."
)

print("\n=== weekly problem ===")
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("AIMCRA weekly", weekly_aimcra)
summarize_economics("MLP weekly (best hard)", weekly_mlp)
print(
    f"weekly MLP minus weekly AIMCRA: "
    f"{scalarize(weekly_mlp['reward']) - scalarize(weekly_aimcra['reward']):+.0f} €/ha"
)

print("\n=== daily deployment of weekly weights ===")
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("AIMCRA daily", standard_practice)
summarize_economics("MLP weekly→daily", daily_mlp)
print(
    f"daily MLP minus daily AIMCRA: "
    f"{scalarize(daily_mlp['reward']) - scalarize(standard_practice['reward']):+.0f} €/ha"
)


In [ ]:
if history:
    hist = pd.DataFrame(history)
    if "phase" in hist.columns:
        weekly_hist = hist[hist["phase"].astype(str).str.startswith("w")]
        if len(weekly_hist):
            hist = weekly_hist
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    axes[0].plot(hist["step"], hist["reward_hard"], color="C0")
    axes[0].axhline(scalarize(weekly_aimcra["reward"]), color="C2", linestyle=":", label="AIMCRA weekly")
    axes[0].set_ylabel("hard R (€ ha$^{-1}$)")
    axes[0].legend()
    axes[1].plot(hist["step"], hist["n_events"], color="C1")
    axes[1].set_ylabel("events")
    axes[1].set_xlabel("Adam step (weekly training)")
    fig.suptitle("Weekly STE training of a random-init MLP", y=0.99)
    fig.tight_layout()
    plt.show()
else:
    print("Checkpoint loaded; weekly training history is not stored in this file.")


## 8. What the weekly-trained MLP does

The learned rule is a closed-loop event policy: on a decision day it reads remainder-to-field-capacity, rain, DVS and calendar, and either applies a 50 mm AIMCRA-class gift (clipped to what the root zone can take) or stays off. It was trained only on Mondays in the WOFOST calendar, `(DOY − 1) mod 7 = 0`. The plots below use those weights **every day**.

AIMCRA remains the agronomic reference: the same gift size, a 35 mm NAP, a 10 mm rain skip, and a 15 September cut-off. Rainfed and potential production bound yield. The economic question is whether the MLP's events beat AIMCRA on $R = 40Y - 25N - I_{\mathrm{mm}}$, not whether they close every last millimetre of drought stress.


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 11), sharex=True)

series = [
    ("rainfed", rainfed_df, "-"),
    ("AIMCRA", standard_practice["frame"], ":"),
    ("MLP weekly→daily", daily_mlp["frame"], "-"),
    ("potential", pp_df, ":"),
]

for name, frame, ls in series:
    axes[0].plot(frame.index, frame["SM"], label=name, linestyle=ls)
    axes[1].plot(frame.index, frame["RFTRA"], label=name, linestyle=ls)
    axes[2].plot(frame.index, frame["LAI"], label=name, linestyle=ls)
    axes[3].plot(frame.index, frame["TWSO"], label=name, linestyle=ls)

mlp_frame = daily_mlp["frame"]
axes[0].bar(
    mlp_frame.index,
    mlp_frame["irrigation"] * 0.004,
    color="C0",
    alpha=0.35,
    width=1.0,
    label="MLP irrigation (scaled mm)",
)
axes[0].axhline(float(provider["SMFCF"]), color="0.5", linewidth=0.7, linestyle=":")
axes[0].set_ylabel("SM $(-)$")
axes[0].legend(loc="upper right", fontsize=8)
axes[1].set_ylabel("RFTRA $(-)$")
axes[2].set_ylabel("LAI $(-)$")
axes[3].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[3].set_xlabel("day")

fig.suptitle("Physiological pathway from irrigation to yield", y=0.99)
fig.tight_layout()
plt.show()


In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(mlp_frame.index, mlp_frame["irrigation"], width=1.0, color="C0", alpha=0.85, label="MLP weekly→daily")
ax.plot(
    standard_practice["frame"].index,
    standard_practice["frame"]["irrigation"],
    color="C2",
    linewidth=0.9,
    alpha=0.85,
    label="AIMCRA",
)
ax.set_ylabel("irrigation (mm day$^{-1}$)")
ax.set_xlabel("day")
ax.legend()
ax.set_title("Daily irrigation events: weekly-trained MLP deployed every day")
fig.tight_layout()
plt.show()


def comparison_row(name, twso, amounts_cm, potential=False):
    y = scalarize(fresh_yield_t_ha(twso))
    if potential:
        return {
            "fresh yield (t/ha)": y,
            "applied irrigation (mm)": 0.0,
            "events": 0,
            "min RFTRA": 1.0,
            "revenue (€/ha)": BEET_PRICE * y,
            "irrigation cost (€/ha)": 0.0,
            "reward (€/ha)": BEET_PRICE * y,
        }
    eco = irrigation_economics(torch.as_tensor(scalarize(twso), dtype=ComputeConfig.get_dtype()), amounts_cm)
    return {
        "fresh yield (t/ha)": y,
        "applied irrigation (mm)": scalarize(eco["applied_mm"]),
        "events": eco["n_events_hard"],
        "min RFTRA": float("nan"),
        "revenue (€/ha)": scalarize(eco["revenue"]),
        "irrigation cost (€/ha)": scalarize(eco["cost"]),
        "reward (€/ha)": scalarize(eco["reward"]),
    }


def with_min_rftra(frame, row_name, table):
    emerged = frame["DVS"] > 0
    table.loc[row_name, "min RFTRA"] = frame.loc[emerged, "RFTRA"].min()
    return table


comparison = pd.DataFrame(
    [
        comparison_row("rainfed", rainfed_df["TWSO"].iloc[-1], zero_amounts),
        comparison_row("standard practice (AIMCRA)", standard_practice["twso"], standard_practice["amounts_cm"]),
        comparison_row("MLP weekly→daily", daily_mlp["twso"], daily_mlp["amounts_cm"]),
        comparison_row("potential production", pp_df["TWSO"].iloc[-1], None, potential=True),
    ],
    index=[
        "rainfed",
        "standard practice (AIMCRA)",
        "MLP weekly→daily",
        "potential production",
    ],
)
comparison = with_min_rftra(rainfed_df, "rainfed", comparison)
comparison = with_min_rftra(standard_practice["frame"], "standard practice (AIMCRA)", comparison)
comparison = with_min_rftra(daily_mlp["frame"], "MLP weekly→daily", comparison)
comparison = with_min_rftra(pp_df, "potential production", comparison)
display(comparison.round(1))


The rainfed crop is the lower bound on yield and a serious economic baseline: it has no irrigation bill. Potential production is beet revenue with a zero irrigation bill by construction. AIMCRA is the grower baseline: a dozen 40 mm-class gifts through mid-September.

The weekly-trained MLP should sit near AIMCRA on both yield and $R$. On the weekly problem it is a fair comparison against weekly AIMCRA (the same handbook, same 7-day grid). Daily deployment of those weights is the operational test against daily AIMCRA. Extra millimetres only pay if they cover `C_event N + c_mm I`.


## 9. Train and test weather years

The MLP was fit on **one** rainfall series: the YAML 2010 weather and soil at 39.30°N, 3.43°W. A closed-loop rule that reads the refill remainder and today's rain should still make sense when 2009 or 2011 rain falls on different days.

Sowing stays 27 March and harvest stays 31 December. Only the campaign year changes. Training weather is the YAML record; test weather is NASA POWER at the same coordinates. AIMCRA is re-evaluated on each year (it has no trained weights). The MLP is frozen at the 2010 weekly checkpoint and deployed daily.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


nasa_weather = NASAPowerWindow(
    SITE_LAT,
    SITE_LON,
    start=dt.date(2008, 1, 1),
    end=dt.date(2012, 12, 31),
)
print(
    f"NASA POWER {SITE_LAT:.5f}°N, {abs(SITE_LON):.5f}°W  "
    f"{nasa_weather.first_date} – {nasa_weather.last_date}"
)

mlp_frozen = IrrigationMLP(period=1)
mlp_frozen.load_state_dict(clone_state(mlp))
mlp_frozen.hard_decisions = True


def crop_window(agro):
    spec = next(iter(agro[0].values()))
    calendar = spec["CropCalendar"]
    return calendar["crop_start_date"], calendar["crop_end_date"]


def season_rain_mm(weather_provider, start, end):
    total = 0.0
    day = start
    one = dt.timedelta(days=1)
    while day <= end:
        total += 10.0 * float(weather_provider(day).RAIN)
        day += one
    return total


def outcome_record(season, rain_mm, controller, outcome, potential=False):
    if potential:
        rec = comparison_row(controller, outcome["twso"], None, potential=True)
    else:
        rec = comparison_row(controller, outcome["twso"], outcome["amounts_cm"])
        emerged = outcome["frame"]["DVS"] > 0
        rec["min RFTRA"] = float(outcome["frame"].loc[emerged, "RFTRA"].min())
    rec["season"] = season
    rec["season rain (mm)"] = rain_mm
    rec["controller"] = controller
    return rec


def evaluate_potential(weather_provider, agro):
    engine = Engine(config=pp_config)
    engine.setup(provider, weather_provider, agro)
    engine.run_till_terminate()
    results = engine.get_output()
    return {"twso": results[-1]["TWSO"], "frame": results_to_frame(results)}


def evaluate_season(season, weather_provider, agro, reuse=None):
    start, end = crop_window(agro)
    rain_mm = season_rain_mm(weather_provider, start, end)
    print(f"\n{season}: season rain {rain_mm:.0f} mm")
    records = []
    specs = [
        ("rainfed", None, wlp_config, False),
        ("standard practice (AIMCRA)", StandardPracticePolicy(), wlp_config, False),
        ("MLP weekly→daily (2010)", mlp_frozen, wlp_config, False),
        ("potential production", None, pp_config, True),
    ]
    for name, ctrl, config, potential in specs:
        if reuse is not None and name in reuse:
            outcome = reuse[name]
        elif potential:
            outcome = evaluate_potential(weather_provider, agro)
        else:
            with torch.no_grad():
                outcome = evaluate_policy(ctrl, provider, weather_provider, agro, config)
        rec = outcome_record(season, rain_mm, name, outcome, potential=potential)
        records.append(rec)
        print(
            f"  {name:28s}  fresh={rec['fresh yield (t/ha)']:5.1f} t/ha  "
            f"applied={rec['applied irrigation (mm)']:5.0f} mm  "
            f"events={int(rec['events']):3d}  "
            f"R={rec['reward (€/ha)']:7.0f} €/ha"
        )
    return records


transfer_rows = []
transfer_rows.extend(
    evaluate_season(
        f"{TRAIN_YEAR} YAML (train)",
        weather,
        agromanagement,
        reuse={
            "rainfed": {
                "twso": rainfed_df["TWSO"].iloc[-1],
                "amounts_cm": zero_amounts,
                "frame": rainfed_df,
            },
            "standard practice (AIMCRA)": standard_practice,
            "MLP weekly→daily (2010)": daily_mlp,
            "potential production": {
                "twso": pp_df["TWSO"].iloc[-1],
                "amounts_cm": zero_amounts,
                "frame": pp_df,
            },
        },
    )
)
for year in TEST_YEARS:
    transfer_rows.extend(
        evaluate_season(
            f"{year} NASA POWER (test)", nasa_weather, agro_for_year(yaml_agro, year)
        )
    )

transfer = pd.DataFrame(transfer_rows)
controller_order = [
    "rainfed",
    "standard practice (AIMCRA)",
    "MLP weekly→daily (2010)",
    "potential production",
]
season_order = [f"{TRAIN_YEAR} YAML (train)"] + [
    f"{year} NASA POWER (test)" for year in TEST_YEARS
]
reward_table = transfer.pivot(index="controller", columns="season", values="reward (€/ha)")
reward_table = reward_table.reindex(
    index=controller_order,
    columns=[name for name in season_order if name in reward_table.columns],
)
print("\nReward (€/ha) by season")
display(reward_table.round(0))
yield_table = transfer.pivot(index="controller", columns="season", values="fresh yield (t/ha)")
yield_table = yield_table.reindex(index=reward_table.index, columns=reward_table.columns)
print("Fresh yield (t/ha) by season")
display(yield_table.round(1))

plot_controllers = [
    "rainfed",
    "standard practice (AIMCRA)",
    "MLP weekly→daily (2010)",
]
ax = reward_table.loc[plot_controllers].T.plot(kind="bar", figsize=(9, 4.2), rot=15)
ax.set_ylabel("reward (€ ha$^{-1}$)")
ax.set_xlabel("")
ax.legend(fontsize=8, loc="upper right")
ax.set_title("Frozen 2010 controllers on YAML train and NASA POWER test years")
fig = ax.get_figure()
fig.tight_layout()
plt.show()


On the 2010 training year the rainfed gap is Mediterranean: this YAML is a dry inland Spanish site.

The transfer question is whether the weekly-trained MLP still fires sensible events when the rain timing changes. AIMCRA will move its NAP threshold with the soil; the MLP can only move if remainder, rain, DVS and day of year still point at a useful gate.


## 10. Challenges, and what this notebook does not solve

Differentiable decision optimization inherits the difficulties of long-horizon control.

**Daily from scratch is not searchable.** A random MLP with a hard $p>0.5$ gate almost never irrigates. STE plus €25 per event then walks remaining ON days to rainfed. Soft daily training learns a high-volume trickle, not AIMCRA's pulse calendar. That is why this notebook trains **weekly** and only then deploys daily.

**Events versus a daily trickle.** AIMCRA advises ~40 mm gifts. The straight-through estimator makes the forward pass look like $\{0,\text{dose}\}$. The backward pass is a biased surrogate: later STE steps can leave a good discrete schedule. Training therefore keeps the best hard $R$, not the last step. Daily STE finetune of weekly weights can beat AIMCRA for a step or two and then collapse; we do not keep that tail.

**Weekly is an inductive bias on time, not an AIMCRA clone.** The MLP is default Linear init. It does not see `RFTRA`. The gift size and refill clip are AIMCRA's, because those are the agronomic action, but the *when* is learned. Weekly AIMCRA is shown only as a diagnostic on the training grid.

**Cost structure.** €1/mm plus €25 per event is an operating-cost split of a hose-reel figure (~€50 per 25 mm), not a farm invoice and not capital recovery. Changing `P`, `c_mm` or `C_event` changes the policy.

**This site is Ciudad Real.** AIMCRA's mean-year net calendar for late-sown beet on a 140–180 mm m$^{-1}$ loam is 14–15 events and ~535–540 mm (May–September). The YAML soil is 166 mm m$^{-1}$. `WaterbalanceFD` has no groundwater. WOFOST `TWSO` is dry biomass (23% DM).

**Transfer.** Weights are fit on 2010 YAML weather and frozen on NASA POWER 2009 and 2011. A stricter protocol would *train* on several years as well. Adding a weather *forecast* to the state is a natural extension; this notebook only uses the current day's rain.

**Reinforcement learning is later.** PPO on the same daily event problem is the obvious next controller class: exploration can invent the first ON days that STE never sees. This notebook stops at differentiable weekly training plus daily deployment.

The point of the example is not an operational irrigation scheduler. It is to show that once WOFOST is differentiable, management can be optimized with the same gradient tape as parameters or hybrid modules, with an interpretable €/ha objective and with consequences that remain visible in the crop's physiological states.
